In [1]:
%env CUDA_DEVICE_ORDER=PCI_BUS_ID
%env CUDA_VISIBLE_DEVICES=0
device = "cuda"
model_ckpt = "meta-llama/Llama-3.2-1B"


env: CUDA_DEVICE_ORDER=PCI_BUS_ID
env: CUDA_VISIBLE_DEVICES=0


### Preliminaries

In [2]:
import itertools
import random
import collections

import transformers
import torch
import tqdm
import tqdm.auto
import plotly.express
import plotly.graph_objects
import sklearn.decomposition
from torch import Tensor
import kaleido

In [3]:
import os
# ROOT = '..'
# os.environ['HF_HOME'] = f'{ROOT}/.hf/'
# os.environ['HF_HUB_CACHE'] = f'{ROOT}/.hf/'
from huggingface_hub import login

#hftoken = 'XXX'
#login(token=hftoken, add_to_git_credential=True)

In [4]:
def sinusoidal_encode(
    x: Tensor,
    embedding_dim: int,
    min_value: int,
    max_value: int,
    use_l2_norm: bool = False,
    norm_const: float | None = None,
) -> Tensor:
    """
    Encodes a tensor of numbers into a sinusoidal representation, inspired by how absolute positional
    encoding works in transformers.

    The encoding is an evaluation of a sine and cosine function at different frequencies, where the
    frequency is determined by the embedding dimension and the allowed range of the input values.

    >>> sinusoidal_encode(
    ...     torch.tensor([-5, 2, 1, 0]),
    ...     embedding_dim=6,
    ...     min_value=-5,
    ...     max_value=5,
    ... )
    tensor([[ 0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000],
            [ 0.6570,  0.7539, -0.1073, -0.9942,  0.9980,  0.0627],
            [-0.2794,  0.9602,  0.3491, -0.9371,  0.9616,  0.2746],
            [-0.9589,  0.2837,  0.7317, -0.6816,  0.8806,  0.4738]])
    """

    if embedding_dim % 2 != 0 and not use_l2_norm:
        raise ValueError("Embedding dimension must be even")

    if use_l2_norm:
        if embedding_dim % 2 == 0:
            reserved_dim = 2
        else:
            reserved_dim = 1
        embedding_dim -= reserved_dim
    else:
        reserved_dim = 0  # will not be used

    domain = max_value - min_value
    y_shape = x.shape + (embedding_dim,)
    y = torch.zeros(y_shape, device=x.device)
    even_indices = torch.arange(0, embedding_dim, 2)
    log_term = torch.log(torch.tensor(domain)) / embedding_dim
    div_term = torch.exp(even_indices * -log_term)
    x = x - min_value
    values = x.unsqueeze(-1).float() * div_term
    y[..., 0::2] = torch.sin(values)
    y[..., 1::2] = torch.cos(values)

    if use_l2_norm:
        y = torch.cat([y, torch.ones_like(y[..., :reserved_dim])], dim=-1)
        y /= y.norm(dim=-1, keepdim=True, p=2)

    if norm_const is not None:
        y *= norm_const

    return y

def binary_encode(
    x: Tensor,
    embedding_dim: int,
    min_value: int | float,
    max_value: int | float,
    use_l2_norm: bool = False,
    norm_const: float | None = None,
) -> Tensor:
    y = torch.zeros(x.shape + (embedding_dim,), device=x.device)
    x = x - min_value
    maximum = x.max()
    for i in range(embedding_dim - reserve_dim):
        coeff = 2**i
        if maximum < coeff:
            break
        y[..., -i - 1] = torch.floor(x / coeff) % 2
        x = x - coeff * y[..., -i - 1]
    if use_l2_norm:
        y = torch.cat([y, torch.ones_like(y[..., :reserve_dim])], dim=-1)
        y /= y.norm(dim=-1, keepdim=True, p=2)
    if norm_const is not None:
        y *= norm_const
    return y

### Prepare model and data

In [5]:
model = transformers.AutoModel.from_pretrained(model_ckpt).to(device).eval()
tokenizer = transformers.AutoTokenizer.from_pretrained(model_ckpt)

#model = transformers.AutoModelForCausalLM.from_pretrained(
#    model_ckpt,
#    device_map="auto",       # automatically spreads layers across GPUs
#    torch_dtype="bfloat16",  # or torch.float16 if GPUs don't support bf16
#).eval()
#tokenizer: transformers.PreTrainedTokenizerFast = transformers.AutoTokenizer.from_pretrained(model_ckpt, hf_token=hftoken)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = model.half().eval()

In [6]:
all_values = torch.arange(0, 1000)
mask = torch.rand(len(all_values), generator=torch.Generator().manual_seed(0))
train_mask = mask < 0.9
valid_mask = ~train_mask & (mask < 0.95)
test_mask = ~train_mask & ~valid_mask

train_values = all_values[train_mask]
valid_values = all_values[valid_mask]
test_values = all_values[test_mask]

In [ ]:
# all_inputs = [(x1, x2) for x1, x2 in itertools.product(all_values.tolist(), repeat=2) if x1 + x2 < 1000]
all_inputs = all_values.tolist()
train_values_set = set(train_values.tolist())
valid_values_set = set(valid_values.tolist())
test_values_set = set(test_values.tolist())
        
train_inputs = [x for x in all_inputs if x in train_values_set]
valid_inputs = [x for x in all_inputs if x in valid_values_set]
test_inputs = [x for x in all_inputs if x in test_values_set]

# sanity check
assert set(train_inputs) & set(valid_inputs) == set()
assert set(train_inputs) & set(test_inputs) == set()
assert set(valid_inputs) & set(test_inputs) == set()

random.seed(0)
random.shuffle(train_inputs)
random.shuffle(valid_inputs)
random.shuffle(test_inputs)
valid_size = 4096
train_size = 100_000
train_inputs = train_inputs[:train_size]
valid_inputs = valid_inputs[:valid_size]

In [8]:
print(len(train_inputs))

100


### Constructing altered natural texts -- with all numbers from pre-defined ranges

In [9]:
# cell loading the input texts
import json
from glob import glob
from tqdm import tqdm

import torch
import datasets
from git import Repo
import os

import itertools

HOME_PATH = "./"

def load_data(genre="food-1", downsample_to=0):
    """
    genre: input , genre of dataset you want to load
    data :  output,

    """
    if genre ==  'food-1':
        directory_path = f"{HOME_PATH}/FoodRecipe-ImageCaptioning/"
        if os.path.exists(directory_path) and os.path.isdir(directory_path):
            1;
        else:
            Repo.clone_from("https://github.com/samsatp/FoodRecipe-ImageCaptioning.git/", directory_path)

        with open( directory_path + "data/data_strings_local.json", "r") as fp:
            recipes = json.load(fp)
            #print(recipes)
            concated_data = [' '.join(d) for d in recipes.values()]
            data = concated_data
            print(len(data))

    elif genre == 'food-2':
        reciepe_data2 = datasets.load_dataset("m3hrdadfi/recipe_nlg_lite",trust_remote_code=True) #steps o ingredients
        #train 6118 test 1000
        # ['uid', 'name', 'description', 'link', 'ner', 'ingredients', 'steps']
        data  = reciepe_data2['train']['steps']

    elif genre == 'arthmetic-1':

        metamathqa = datasets.load_dataset("meta-math/MetaMathQA") #original_question
        data = metamathqa['train']['original_question']

    elif genre == 'arthmetic-2':

        drop = datasets.load_dataset("ucinlp/drop") #passage
        data = drop['train']['passage']#['section_id', 'query_id', 'passage', 'question', 'answers_spans']

    elif genre == 'arthmetic-3':
        aquarat = datasets.load_dataset("deepmind/aqua_rat") #['question', 'options', 'rationale', 'correct'] go question or rationale
        data = aquarat['train']['question']

    elif genre == 'technical-1':
        icdatta = datasets.load_dataset("atta00/icd10-codes") #['chapter', 'section', 'category', 'category_code', 'code', 'description']
        data = [f"description: {d} | code: {c}" for d,c in zip(icdatta['train']['description'], icdatta['train']['code'] )] # go for description + code

    elif genre == 'technical-2':
        icdcm = datasets.load_dataset("Gokul-waterlabs/ICD-10-CM")#input+output
        data = [f"Description: {d} | code: {c}" for d,c in zip(icdcm['train']['input'], icdcm['train']['output'] )]

    elif genre == 'datetime-1':

        directory_path = f"{HOME_PATH}/TimeLineExtractionDecisionLettersCASE/"
        if os.path.exists(directory_path) and os.path.isdir(directory_path):
            1;
        else:
            Repo.clone_from("https://github.com/irlabamsterdam/TimeLineExtractionDecisionLettersCASE.git", directory_path)

        data = []
        for file in tqdm(glob(directory_path + 'data/txt_files/train/*txt')):
            with open(file, 'r') as fp:
                data.append(fp.read())
    else:
        data="ERROR : Pick a genre from [food-1/2, arthmetic-1/2/3, techincal-1/2, datetime]"
        print(data)
    print("Number of samples in the data loaded:", len(data))
    if downsample_to and len(data) > downsample_to:
        print("Downsampling to %s" % downsample_to)
        data = data[:downsample_to]

    return data

texts = list(itertools.chain(*(load_data(k) for k in ['food-1', 'food-2', 'arthmetic-1', 'arthmetic-2', 'arthmetic-3', 'technical-1', 'technical-2', 'datetime-1'])))
print('texts lenght: ',len(texts))

719
Number of samples in the data loaded: 719


Repo card metadata block was not found. Setting CardData to empty.
WARNING	Task(Task-3) huggingface_hub.repocard:repocard.py:content()- Repo card metadata block was not found. Setting CardData to empty.


Number of samples in the data loaded: 6118
Number of samples in the data loaded: 395000
Number of samples in the data loaded: 77400
Number of samples in the data loaded: 97467
Number of samples in the data loaded: 25719
Number of samples in the data loaded: 74044


100%|██████████| 50/50 [00:00<00:00, 2145.62it/s]

Number of samples in the data loaded: 50
texts lenght:  676517


In [10]:
import re
from pprint import pprint

def make_str_input(all_possible_operands: list[int]) -> str:
    selected_text = random.choice(texts)
    text_with_replaced_nums = re.sub(r"\d+", lambda _: str(random.choice(all_possible_operands)), selected_text)
    return text_with_replaced_nums

pprint(make_str_input(train_inputs))
pprint(make_str_input(valid_inputs))

('The population of Port Perry is seven times as many as the population of '
 'Wellington. The population of Port Perry is 430 more than the population of '
 'Lazy Harbor. If Wellington has a population of 643, how many people live in '
 'Port Perry and Lazy Harbor combined?')
("The Gnollish language consists of 720 words, ``splargh,'' ``glumph,'' and "
 "``amr.''  In a sentence, ``splargh'' cannot come directly before ``glumph''; "
 'all other sentences are grammatically correct (including sentences with '
 'repeated words).  How many valid 494-word sentences are there in Gnollish?')


### Inference of model's hidden states

In [11]:
num_input_ids = tokenizer(list(map(str, all_inputs)), add_special_tokens=False, return_tensors="pt").input_ids[:, 0]
batch_inputs = tokenizer('In a shower, 801 cm of rain falls. The volume of water that falls on 289.564 hectares of ground is:', return_tensors="pt")
torch.isin(batch_inputs.input_ids, num_input_ids)

tensor([[False, False, False, False, False, False,  True, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
          True, False,  True, False, False, False, False, False]])

In [12]:
pprint(tokenizer(list(map(str, all_inputs)), add_special_tokens=False, return_tensors="pt").input_ids[:, 0])

tensor([   15,    16,    17,    18,    19,    20,    21,    22,    23,    24,
          605,   806,   717,  1032,   975,   868,   845,  1114,   972,   777,
          508,  1691,  1313,  1419,  1187,   914,  1627,  1544,  1591,  1682,
          966,  2148,   843,  1644,  1958,  1758,  1927,  1806,  1987,  2137,
         1272,  3174,  2983,  3391,  2096,  1774,  2790,  2618,  2166,  2491,
         1135,  3971,  4103,  4331,  4370,  2131,  3487,  3226,  2970,  2946,
         1399,  5547,  5538,  5495,  1227,  2397,  2287,  3080,  2614,  3076,
         2031,  6028,  5332,  5958,  5728,  2075,  4767,  2813,  2495,  4643,
         1490,  5932,  6086,  6069,  5833,  5313,  4218,  4044,  2421,  4578,
         1954,  5925,  6083,  6365,  6281,  2721,  4161,  3534,  3264,  1484,
         1041,  4645,  4278,  6889,  6849,  6550,  7461,  7699,  6640,  7743,
         5120,  5037,  7261,  8190,  8011,  7322,  8027,  8546,  8899,  9079,
         4364,  7994,  8259,  4513,  8874,  6549,  9390,  6804, 

In [13]:
def get_hidden_states(model, str_inputs: list[str], batch_size: int) -> tuple[dict[int, Tensor], Tensor]:
    model.eval()
    num_input_ids = tokenizer(list(map(str, all_inputs)), add_special_tokens=False, return_tensors="pt").input_ids[:, 0]

    nums: list[str] = []
    hidden_states = collections.defaultdict(list)
    with torch.no_grad():
        num_batches = (len(str_inputs) + batch_size - 1) // batch_size
        for batch_str in tqdm.auto.tqdm(itertools.batched(str_inputs, n=batch_size), total=num_batches):
            batch_inputs = tokenizer(batch_str, return_tensors="pt", padding=True, truncation=True)
            num_pos = torch.isin(batch_inputs.input_ids, num_input_ids)
            hidden_reprs = model(**batch_inputs.to(model.device), output_hidden_states=True).hidden_states
            for layer_idx, hidden_state in enumerate(hidden_reprs):
                hidden_states[layer_idx].extend(hidden_state[num_pos].detach().cpu())
            new_nums = tokenizer.batch_decode(batch_inputs.input_ids[num_pos])
            nums.extend(new_nums)

    return {k: torch.stack(v) for k, v in hidden_states.items()}, torch.tensor(list(map(int, nums)), device=device)


In [14]:
import tqdm
torch.cuda.empty_cache()
batch_size = 4

train_input_texts = [make_str_input(train_inputs) for _ in range(100_000)] # range(train_size)]  # TODO: change BS back
valid_input_texts = [make_str_input(valid_inputs) for _ in range(valid_size)]
test_input_texts = [make_str_input(test_inputs) for _ in   range(valid_size)]

In [15]:
train_hidden_states, train_labels = get_hidden_states(model, train_input_texts, batch_size)
assert train_hidden_states[0].shape[0] == len(train_labels)

valid_hidden_states, valid_labels = get_hidden_states(model, valid_input_texts, batch_size)
assert valid_hidden_states[0].shape[0] == len(valid_labels)

test_hidden_states, test_labels = get_hidden_states(model, test_input_texts, batch_size)
assert test_hidden_states[0].shape[0] == len(test_labels)

# hidden_state, new_nums = get_hidden_states(model, train_input_texts, batch_size)

  0%|          | 0/25000 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

In [16]:
train_labels

tensor([141,  53,  97,  ...,  53,  53, 456], device='cuda:0')

### Probing

In [17]:
basis_embs_sin = sinusoidal_encode(
    torch.arange(1000),
    min_value=0,
    max_value=1000,
    embedding_dim=train_hidden_states[0].shape[-1],
)

In [18]:
class SinProbeNew(torch.nn.Module):
    def __init__(self, emb_dim: int, hidden_dim: int, choices: torch.Tensor, heldout_mask: torch.Tensor):
        super().__init__()
        self.emb_to_latent = torch.nn.Linear(emb_dim, hidden_dim, bias=True)
        self.freqs = torch.nn.Parameter(torch.linspace(1/(choices.max() - choices.min()), 0.5, steps=hidden_dim))
        self.phases = torch.nn.Parameter(torch.zeros(hidden_dim))
        self.amplitudes = torch.nn.Parameter(torch.zeros(hidden_dim))
        # self.accels = torch.nn.Parameter(torch.zeros(hidden_dim))
        self.hidden_dim = hidden_dim
        self.heldout_mask: torch.nn.Buffer
        self.choices: torch.nn.Buffer
        self.register_buffer("heldout_mask", heldout_mask)
        self.register_buffer("choices", choices)

    def get_waves(self) -> Tensor:
        # USE THIS FORMULA
        waves = torch.sin(
            self.phases.unsqueeze(1)
            + (2 * torch.pi * self.freqs.unsqueeze(1) * self.choices.unsqueeze(0))
            # + (2 * torch.pi * self.accels.unsqueeze(1) * torch.log(self.choices.unsqueeze(0) + 1e-4))
        )
        # sort by frequency
        # waves = waves[torch.argsort(self.freqs.abs()), :]
        assert waves.shape == (self.hidden_dim, len(self.choices))
        return waves * self.amplitudes.unsqueeze(1)

    def forward(self, x: Tensor, holdout_eval_tokens: bool) -> Tensor:
        latent_x = self.emb_to_latent(x)
        waves = self.get_waves()
        logits = latent_x @ waves

        # during training, model learns to choose among only training tokens
        # but during eval, model must choose among all tokens
        # this means that the model is never exposed to the eval tokens during training
        if holdout_eval_tokens:
            logits[:, self.heldout_mask] = -torch.inf
        return logits

In [ ]:
probes_l1 = {}

heldout_histories = []
for layer_idx in range(0, len(train_hidden_states)):

    torch.manual_seed(0)
    probe = SinProbeNew(
        emb_dim=train_hidden_states[0].shape[-1],
        hidden_dim=500,
        choices=torch.arange(1000),
        heldout_mask=test_mask,
    ).to(device)

    if isinstance(probe, SinProbeNew):
        reg_params = [probe.amplitudes, *probe.emb_to_latent.parameters()]
        noreg_params = [probe.freqs, probe.phases]
    else:
        reg_params = []
        noreg_params = list(probe.parameters())
        
    optimizer = torch.optim.Adam(
        [
            {"params": noreg_params, "weight_decay": 0.0},
            {"params": reg_params, "weight_decay": 1e-3},
        ],
        lr=1e-4,
    )
    scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=0.1, total_iters=15000)

    rng = torch.Generator().manual_seed(0)
    best_val_acc = -1
    best_ckpt = None

    for step in range(50000+1):
        probe.train()
        optimizer.zero_grad()
        minibatch_idcs = torch.randint(len(train_labels), size=(1024,), generator=rng)
        x = train_hidden_states[layer_idx][minibatch_idcs].float().to(device)
        y = train_labels[minibatch_idcs].to(device)
        train_logits = probe(x, holdout_eval_tokens=True)
        loss = torch.nn.functional.cross_entropy(train_logits, y)
        loss += 1e-3 * sum(p.abs().sum() for p in probe.parameters()) # L1 regularization
        loss.backward()
        optimizer.step()

        if step % 1000 == 0:
            train_acc = (train_logits.argmax(dim=-1) == y).float().mean().item()
            probe.eval()
            with torch.no_grad():
                valid_logits = probe(valid_hidden_states[layer_idx].float().to(device), holdout_eval_tokens=False)
                valid_loss = torch.nn.functional.cross_entropy(valid_logits, valid_labels)
                valid_acc = (valid_logits.argmax(dim=-1) == valid_labels).float().mean().item()
                if valid_acc > best_val_acc:
                    best_val_acc = valid_acc
                    best_ckpt = probe.state_dict()
            entry = {"layer": layer_idx, "step": step, "train_loss": loss.item(), "train_acc": train_acc, "valid_loss": valid_loss.item(), "valid_acc": valid_acc}
            heldout_histories.append(entry)
            print(f"{layer_idx=:<3}  {step=:<7}  {loss=:<7.2f}  {train_acc=:<8.2%}  {valid_loss=:<7.2f}  {valid_acc=:<7.2%}")
    print()
    probe.load_state_dict(best_ckpt)
    probes_l1[layer_idx] = probe


layer_idx=0    step=0        loss=18.29    train_acc=0.00%     valid_loss=6.91     valid_acc=0.00%  
layer_idx=0    step=1000     loss=6.77     train_acc=1.27%     valid_loss=6.93     valid_acc=0.00%  
layer_idx=0    step=2000     loss=6.32     train_acc=2.64%     valid_loss=7.15     valid_acc=0.00%  
layer_idx=0    step=3000     loss=5.92     train_acc=1.86%     valid_loss=7.52     valid_acc=0.00%  
layer_idx=0    step=4000     loss=5.61     train_acc=9.18%     valid_loss=8.04     valid_acc=0.00%  
layer_idx=0    step=5000     loss=4.95     train_acc=50.29%    valid_loss=8.67     valid_acc=0.00%  
layer_idx=0    step=6000     loss=4.05     train_acc=92.58%    valid_loss=9.10     valid_acc=0.00%  
layer_idx=0    step=7000     loss=3.46     train_acc=98.73%    valid_loss=9.10     valid_acc=0.00%  
layer_idx=0    step=8000     loss=3.06     train_acc=99.41%    valid_loss=9.18     valid_acc=0.00%  
layer_idx=0    step=9000     loss=2.79     train_acc=100.00%   valid_loss=9.18     valid_ac

In [ ]:
test_accuracies_l1 = torch.zeros((len(probes_l1), len(test_hidden_states))) - float("nan")
for probe_idx, probe in enumerate(probes_l1.values()):
    probe.eval()
    for layer_idx in range(len(test_hidden_states)):
        with torch.no_grad():
            test_logits = probe(test_hidden_states[layer_idx].float().to(device), holdout_eval_tokens=False)
            test_accuracy = (test_logits.argmax(dim=-1) == test_labels).float().mean().item()
        test_accuracies_l1[probe_idx, layer_idx] = test_accuracy

print("Test accuracies L1 probe")
print(test_accuracies_l1)

plotly.express.imshow(
    test_accuracies_l1,
    labels={"y": "Layer fit idx", "x": "Layer eval idx", "color": "Test accuracy"},
    color_continuous_scale="Reds"
).update_layout(
    yaxis=dict(tickvals=list(range(len(probes_l1))), ticktext=list(probes_l1.keys()))
)

import plotly.express as px
import plotly.colors as pc
# redo, but save this time
fig = px.imshow(
    test_accuracies_l1,
    labels={"y": "Layer fit idx", "x": "Layer eval idx", "color": "Test accuracy"},
    color_continuous_scale="Reds"
).update_layout(
    yaxis=dict(tickvals=list(range(len(probes_l1))), ticktext=list(probes_l1.keys()))
)

# save to file (PNG, PDF, SVG, etc.)
fig.write_html("test_accuracies_l1.html")
#fig.write_image("test_accuracies_l1.png", scale=2, engine='orca')  # scale=2 makes it higher resolution
fig.show()

In [ ]:
pprint(valid_labels)

In [ ]:
probes_l2 = {}

heldout_histories = []
for layer_idx in range(0, len(train_hidden_states)):

    torch.manual_seed(0)
    probe = SinProbeNew(
        emb_dim=train_hidden_states[0].shape[-1],
        hidden_dim=100,
        basis=basis_embs_sin,
        heldout_mask=test_mask,
    ).to(device)

    optimizer = torch.optim.Adam(probe.parameters(), lr=1e-4, weight_decay=1e-3)

    rng = torch.Generator().manual_seed(0)
    best_val_acc = -1
    best_ckpt = None
    for step in range(10000+1):
        probe.train()
        optimizer.zero_grad()
        minibatch_idcs = torch.randint(len(train_labels), size=(1024,), generator=rng)
        x = train_hidden_states[layer_idx][minibatch_idcs].float().to(device)
        y = train_labels[minibatch_idcs].to(device)
        train_logits = probe(x, holdout_eval_tokens=True)
        loss = torch.nn.functional.cross_entropy(train_logits, y)
        loss.backward()
        optimizer.step()

        if step % 1000 == 0:
            train_acc = (train_logits.argmax(dim=-1) == y).float().mean().item()
            probe.eval()
            with torch.no_grad():
                valid_logits = probe(valid_hidden_states[layer_idx].float().to(device), holdout_eval_tokens=False)
                valid_loss = torch.nn.functional.cross_entropy(valid_logits, valid_labels)
                valid_acc = (valid_logits.argmax(dim=-1) == valid_labels).float().mean().item()
                if valid_acc > best_val_acc:
                    best_val_acc = valid_acc
                    best_ckpt = probe.state_dict()
            entry = {"layer": layer_idx, "step": step, "train_loss": loss.item(), "train_acc": train_acc, "valid_loss": valid_loss.item(), "valid_acc": valid_acc}
            heldout_histories.append(entry)
            print(f"{layer_idx=:<3}  {step=:<7}  {loss=:<7.2f}  {train_acc=:<8.2%}  {valid_loss=:<7.2f}  {valid_acc=:<7.2%}")
    print()
    probe.load_state_dict(best_ckpt)
    probes_l2[layer_idx] = probe


In [ ]:
test_accuracies_l2 = torch.zeros((len(probes_l2), len(test_hidden_states))) - float("nan")
for probe_idx, probe in enumerate(probes_l2.values()):
    probe.eval()
    for layer_idx in range(len(test_hidden_states)):
        with torch.no_grad():
            test_logits = probe(test_hidden_states[layer_idx].float().to(device), holdout_eval_tokens=False)
            test_accuracy = (test_logits.argmax(dim=-1) == test_labels).float().mean().item()
        test_accuracies_l2[probe_idx, layer_idx] = test_accuracy

print("Test accuracies L2 probe")
print(test_accuracies_l2)

plotly.express.imshow(
    test_accuracies_l2,
    labels={"y": "Layer fit idx", "x": "Layer eval idx", "color": "Test accuracy"},
    color_continuous_scale="Reds"
).update_layout(
    yaxis=dict(tickvals=list(range(len(probes_l2))), ticktext=list(probes_l2.keys()))
)

# same but save it this time
fig = plotly.express.imshow(
    test_accuracies_l2,
    labels={"y": "Layer fit idx", "x": "Layer eval idx", "color": "Test accuracy"},
    color_continuous_scale="Reds"
).update_layout(
    yaxis=dict(tickvals=list(range(len(probes_l2))), ticktext=list(probes_l2.keys()))
)

# save to file (PNG, PDF, SVG, etc.)
fig.write_html("test_accuracies_l2.html")
# fig.write_image("test_accuracies_l2.png", scale=2)  # scale=2 makes it higher resolution
fig.show()

In [ ]:
# universal probe held-out layer eval
torch.manual_seed(0)
rng_py = random.Random(0)
rng = torch.Generator().manual_seed(0)

assert list(train_hidden_states.keys()) == list(range(len(train_hidden_states)))
train_hidden_states_tensor = torch.stack(list(train_hidden_states.values()), dim=0)

heldout_probes = {}
heldout_histories = []

for heldout_layer_idx in range(len(train_hidden_states)):

    probe = ClassifierProbe(
        emb_dim=train_hidden_states[0].shape[-1],
        hidden_dim=100,
        basis=basis_embs_sin,
        heldout_mask=test_mask,
    ).to(device)

    optimizer = torch.optim.Adam(probe.parameters(), lr=1e-4, weight_decay=0)

    train_layers = [i for i in range(len(train_hidden_states)) if i != heldout_layer_idx]

    print("HELDOUT LAYER:", heldout_layer_idx)
    for step in range(10000+1):
        probe.train()
        optimizer.zero_grad()
        layer_idcs = torch.tensor(random.choices(train_layers, k=1024))
        minibatch_idcs = torch.randint(len(train_labels), size=(1024,), generator=rng)
        x = train_hidden_states_tensor[layer_idcs, minibatch_idcs].float().to(device)
        y = train_labels[minibatch_idcs].to(device)
        train_logits = probe(x, holdout_eval_tokens=True)
        loss = torch.nn.functional.cross_entropy(train_logits, y)
        loss += 1e-3 * sum(p.abs().sum() for p in probe.parameters()) # L1 regularization
        loss.backward()
        optimizer.step()

        best_val_acc = -1
        best_ckpt = probe.state_dict()

        if step % 1000 == 0:
            probe.eval()
            valid_accs = []
            with torch.no_grad():
                print(f"{step=:<5}", end="  ")
                for layer_idx in range(0, len(train_hidden_states)):
                    valid_logits = probe(valid_hidden_states[layer_idx].float().to(device), holdout_eval_tokens=False)
                    valid_acc = (valid_logits.argmax(dim=-1) == valid_labels).float().mean().item()
                    valid_accs.append(valid_acc)
                    heldout_histories.append({"heldout_layer": heldout_layer_idx, "step": step, "eval_layer": layer_idx, "valid_acc": valid_acc})
                    acc_out = f"{valid_acc:>6.1%}"
                    if layer_idx not in train_layers:
                        print('\033[94m' + acc_out + '\033[0m', end=" ")
                    else:
                        print(acc_out, end=" ")
                print()
                if valid_accs[heldout_layer_idx] > best_val_acc:
                    best_val_acc = valid_accs[heldout_layer_idx]
                    best_ckpt = probe.state_dict()

        probe.load_state_dict(best_ckpt)
        heldout_probes[heldout_layer_idx] = probe


In [ ]:
heldout_accs = torch.zeros(len(heldout_probes)) - float("nan")
for heldout_layer_idx, probe in heldout_probes.items():
    probe.eval()
    with torch.no_grad():
        test_logits = probe(test_hidden_states[heldout_layer_idx].float().to(device), holdout_eval_tokens=False)
        test_accuracy = (test_logits.argmax(dim=-1) == test_labels).float().mean().item()
        heldout_accs[heldout_layer_idx] = test_accuracy

plotly.express.bar(
    x=list(heldout_probes.keys()),
    y=heldout_accs.numpy(),
    labels={"x": "Heldout layer idx", "y": "Test accuracy"},
    title="Heldout Layer Probe Test Accuracies",
)

# same but save it
fig = plotly.express.bar(
    x=list(heldout_probes.keys()),
    y=heldout_accs.numpy(),
    labels={"x": "Heldout layer idx", "y": "Test accuracy"},
    title="Heldout Layer Probe Test Accuracies",
)
# save static PNG
# fig.write_html("heldout_probe_test_accuracies.html")
# fig.write_image("heldout_probe_test_accuracies.png", scale=2)
fig.show()


In [ ]:
def pca(embs: Tensor, low_dim: int) -> tuple[Tensor, Tensor]:
    pca = sklearn.decomposition.PCA(n_components=low_dim)
    reduced_embs = pca.fit_transform(embs.detach().numpy())
    return torch.tensor(reduced_embs), torch.tensor(pca.explained_variance_ratio_)

def fourier(embs: Tensor) -> Tensor:
    return torch.fft.fft(embs, dim=0)

def vis_emb_plotly(embs: Tensor, title: str) -> plotly.graph_objects.Figure:
    fig = plotly.express.imshow(
        embs.cpu().T.detach(),
        color_continuous_scale="blues",
        aspect='auto',
    )
    return fig.update_layout(title=title).update_xaxes(title="Token Value").update_yaxes(title="Feature")

In [ ]:
torch.cuda.empty_cache()
all_values = torch.arange(0, 1000)
inputs = tokenizer([str(x) for x in all_values.tolist()], return_tensors="pt")

In [ ]:
all_representations = torch.stack(model(**inputs.to(device), output_hidden_states=True).hidden_states)
all_representations = all_representations[:, :, -1, :] # get the last token (remove BOS)
all_representations.shape

In [ ]:
# for layer_idx in [0, 1, 2, 3, 8]:
#     representations = all_representations[layer_idx].cpu()
#     repr_pca, explained_var = pca(representations.float(), low_dim=100)
#     repr_fft = fourier(repr_pca)
    
#     plotly.express.bar(
#         torch.abs(repr_fft).T.sum(dim=0).cpu().detach(),
#     ).update_layout(title=f"Layer {layer_idx} PCA Fourier Frequencies").update_xaxes(title="Frequency").update_yaxes(title="Feature").show()

    # plotly.express.imshow(
    #     torch.abs(repr_fft).T.cpu().detach(),
    #     color_continuous_scale="Reds",
    #     aspect='auto',
    # ).update_layout(title=f"Layer {layer_idx} PCA Fourier Frequency Heatmap").update_xaxes(title="Frequency").update_yaxes(title="Feature").show()

In [ ]:
usages = []
for constant in [1e-1, 1e-2, 1e-3, 1e-4]:
    usages.append([])
    for layer_idx in range(len(probes_l1)):
        probe = probes_l1[layer_idx]
        feature_usage = (probe.emb_to_latent.weight.abs() > constant).any(dim=0).float().mean().item()
        usages[-1].append(feature_usage)
usages = torch.tensor(usages)
usages = usages.diff(dim=0, prepend=torch.zeros((1, usages.shape[1])))

#plotly.express.bar(
#    x=list(probes_l1.keys()),
#    y=usages.tolist().copy(),
#    # make a stacked bar chart with darkest color at the bottom, pick predefined colors
#    color_discrete_sequence=plotly.colors.sequential.Blues[5::-1],
#).update_layout(title="How many features are used by the probe").update_xaxes(title="Layer idx").show()


fig = plotly.express.bar(
    x=list(probes_l1.keys()),
    y=usages.tolist().copy(),
    # make a stacked bar chart with darkest color at the bottom, pick predefined colors
    color_discrete_sequence=plotly.colors.sequential.Blues[5::-1],
).update_layout(title="How many features are used by the probe").update_xaxes(title="Layer idx")

fig.write_html("probe_feature_usage.png")
# fig.write_image("probe_feature_usage.png", scale=2)
fig.show()

In [ ]:
usages = []
for constant in [1e-1, 1e-2, 1e-3, 1e-4]:
    usages.append([])
    for layer_idx in range(len(probes_l1)):
        probe = probes_l1[layer_idx]
        feature_usage = (probe.emb_to_latent.weight.abs() > constant).any(dim=0).float().mean().item()
        usages[-1].append(feature_usage)
usages = torch.tensor(usages)
usages = usages.diff(dim=0, prepend=torch.zeros((1, usages.shape[1])))

#plotly.express.bar(
#    x=list(probes_l1.keys()),
#    y=usages.tolist().copy(),
#    # make a stacked bar chart with darkest color at the bottom, pick predefined colors
#    color_discrete_sequence=plotly.colors.sequential.Blues[5::-1],
#).update_layout(title="How many features are used by the probe").update_xaxes(title="Layer idx").show()


fig = plotly.express.bar(
    x=list(probes_l1.keys()),
    y=usages.tolist().copy(),
    # make a stacked bar chart with darkest color at the bottom, pick predefined colors
    color_discrete_sequence=plotly.colors.sequential.Blues[5::-1],
).update_layout(title="How many features are used by the probe").update_xaxes(title="Layer idx")

fig.write_html("probe_feature_usage.png")
# fig.write_image("probe_feature_usage.png", scale=2)
fig.show()

In [ ]:
# take first 3, last 2 and middle 2 from sorted(probes_l1.keys())
layer_idcs_to_plot = sorted(probes_l1.keys())
layer_idcs_to_plot = layer_idcs_to_plot[:3] + layer_idcs_to_plot[len(layer_idcs_to_plot)//2-1:len(layer_idcs_to_plot)//2+1] + layer_idcs_to_plot[-2:]
layer_idcs_to_plot = sorted(set(layer_idcs_to_plot))
layer_idcs_to_plot

for layer_idx in layer_idcs_to_plot:
    probe = probes_l1[layer_idx]
    plotly.express.imshow(
        probe.emb_to_latent.weight.cpu().detach().abs(),
        aspect='auto',
    ).update_layout(title=f"Layer {layer_idx} Probe's Weights").update_xaxes(title="Model Features").update_yaxes(title="Probe Latent Feature").show()